# Exploração das Sessões de Recarga

Este notebook explora os dados simulados gerados pelo carregador GoodWe HCA G2 para entender os padrões de consumo e validar os modelos de IA do projeto EV ChargeOps.

In [ ]:
from pathlib import Path
import pandas as pd

# Configuração do caminho para carregar os dados de forma robusta
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sessions = pd.read_csv(project_root / 'data' / 'sessoes.csv')
sessions.head()

In [ ]:
sessions.describe(include='all')

## 1. Visualização de Dados
Análise gráfica para entender a distribuição de consumo e justificar a necessidade da tarifa dinâmica.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configura o estilo dos gráficos
sns.set_theme(style="whitegrid")

# Gráfico: Distribuição de Consumo (kWh)
plt.figure(figsize=(10, 4))
sns.histplot(sessions['kwh_consumed'], bins=20, kde=True, color='blue')
plt.title('Distribuição de Energia Consumida (kWh) por Sessão')
plt.xlabel('kWh Consumido')
plt.ylabel('Frequência')
plt.show()

## 2. Prototipação da IA (Detecção de Anomalias)
Testando o algoritmo Isolation Forest para identificar possíveis fraudes, falhas no equipamento ou consumos irreais antes de fechar o faturamento.

In [ ]:
from sklearn.ensemble import IsolationForest

print("Testando o modelo de Detecção de Anomalias (Isolation Forest)...")
modelo = IsolationForest(contamination=0.05, random_state=42)

# Treina o modelo com duração e consumo
sessions['anomalia'] = modelo.fit_predict(sessions[['duration_hours', 'kwh_consumed']])

# Separa as anomalias (-1) do comportamento normal (1)
anomalias = sessions[sessions['anomalia'] == -1]
normais = sessions[sessions['anomalia'] == 1]

# Gráfico de dispersão mostrando as anomalias em vermelho
plt.figure(figsize=(10, 5))
plt.scatter(normais['duration_hours'], normais['kwh_consumed'], c='blue', label='Normal', alpha=0.6)
plt.scatter(anomalias['duration_hours'], anomalias['kwh_consumed'], c='red', label='Anomalia (Suspeito)', s=100)
plt.title('Detecção de Fraude/Falha nas Recargas')
plt.xlabel('Duração (Horas)')
plt.ylabel('Consumo (kWh)')
plt.legend()
plt.show()

print(f"Total de sessões normais: {len(normais)}")
print(f"Total de anomalias detectadas: {len(anomalias)}")

## 3. Conclusões da Análise
Através da análise exploratória, confirmamos a variação na distribuição de consumo. A aplicação do **Isolation Forest** conseguiu identificar com sucesso as sessões que fogem do padrão (muito tempo conectado com baixo consumo, ou picos irreais de kWh). Isso valida a inclusão deste modelo de IA no orquestrador principal do EV ChargeOps para auditoria prévia ao cálculo de rateio.